## Historical Archive Notice

This notebook is preserved for chronology and debugging history only. It is not the official midterm-compliant reproduction or submission path. Use `scripts/train_raw_baseline.py` and `notebooks/kaggle_submit_raw_baseline.ipynb` instead.


#Training Code

In [1]:
# Install required packages (uncomment if needed)
!pip install transformers peft accelerate bitsandbytes datasets trl cairosvg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ---------- 1. Model & Tokenizer ----------
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # Good balance of size and capability

# QLoRA: load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in BF16
    bnb_4bit_use_double_quant=True,         # Double quantization
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Prepare model for k-bit training (freezes base, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_ID}")
print(f"Model dtype: {model.dtype}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Model dtype: torch.float32
Total parameters: 888,616,448


In [ ]:
# ---------- 2. LoRA Configuration ----------
lora_config = LoraConfig(
    r=16,                     # Rank — start with 16, increase if underfitting
    lora_alpha=32,            # Scaling: effective lr multiplier = alpha/r = 2.0
    lora_dropout=0.05,        # Light regularization
    target_modules=[          # Which linear layers to add LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # MLP (feed-forward)
    ],
    bias="none",              # Don't train biases
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable vs. total parameters
model.print_trainable_parameters()
# Output example: "trainable params: 6,815,744 || all params: 1,549,507,584 || trainable%: 0.4399"

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
# ---------- 3. Inspecting LoRA Layers ----------
# Let's see what LoRA actually added to the model

print("Layers with LoRA adapters:\n")
for name, module in model.named_modules():
    if "lora" in name.lower() and hasattr(module, 'weight'):
        print(f"  {name}: {module.weight.shape}")

Layers with LoRA adapters:

  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default: torch.Size([1536, 16])
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default: torch.Size([1536, 16])
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default: torch.Size([8960, 16])
  base_model.model.model.layers.0.mlp.up_proj.lora_A.default: torch.Size([16, 15

In [ ]:
# ---------- 4. Dataset Preparation ----------
def format_svg_sample(prompt: str, svg_code: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n{svg_code}"

# Example
sample = format_svg_sample(
    prompt="A red circle centered on a white background",
    svg_code='<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100"><circle cx="50" cy="50" r="40" fill="red"/></svg>'
)
print("Formatted training sample:")
print(sample[:500])

Formatted training sample:
Prompt: A red circle centered on a white background
SVG:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100"><circle cx="50" cy="50" r="40" fill="red"/></svg>


In [ ]:
# ---------- 5. Training Setup (using SFTTrainer from TRL) ----------
from transformers import TrainingArguments

from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(
    output_dir="/content/drive/MyDrive/DL Midterm/svg-lora-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=1000,
    eval_strategy="no",
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,
    max_grad_norm=0.3,
    report_to="none",
    max_length=1024,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

csv_path = "/content/drive/MyDrive/DL Midterm/train.csv"
raw_dataset = load_dataset("csv", data_files=csv_path)["train"]

def is_valid_row(example):
    return (
        example.get("prompt") is not None
        and example.get("svg") is not None
        and str(example["prompt"]).strip() != ""
        and str(example["svg"]).strip() != ""
    )

def to_training_text(example):
    return {
        "text": format_svg_sample(
            prompt=str(example["prompt"]).strip(),
            svg_code=str(example["svg"]).strip()
        )
    }

dataset = raw_dataset.filter(is_valid_row)
dataset = dataset.map(to_training_text, remove_columns=raw_dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset[0]["text"][:1000])

# Optional length sanity check
sample_count = min(100, len(train_dataset))
lengths = [
    len(tokenizer(train_dataset[i]["text"])["input_ids"])
    for i in range(sample_count)
]
print("Max token length:", max(lengths))
print("Avg token length:", sum(lengths) / len(lengths))

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Prompt: A black square with a white checkmark inside it.
SVG:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#333333" fill-opacity="1.0"  filling="0" d="M157.4340057373047 25.451995849609375 L42.23400115966797 25.451995849609375 C32.824005126953125 25.451995849609375 25.16699981689453 33.108001708984375 25.16699981689453 42.51900100708008 L25.16699981689453 157.718994140625 C25.16699981689453 167.12899780273438 32.824005126953125 174.78500366210938 42.23400115966797 174.78500366210938 L157.4340057373047 174.78500366210938 C166.843994140625 174.78500366210938 174.50100708007812 167.12899780273438 174.50100708007812 157.718994140625 L174.50100708007812 42.51900100708008 C174.50100708007812 33.108001708984375 166.843994140625 25.451995849609375 157.4340057373047 25.451995849609375 Z M145.2030029296875 73.072998046875 L90.47100067138672 127.80400085449219 C88.26699829101562 130.00900268554688 84.63999938964844 130.00900268

In [ ]:
# ---------- 6. Training Loop (reference — requires a dataset) ----------

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset, # Your HuggingFace Dataset object
    eval_dataset=eval_dataset,
    processing_class=tokenizer, # SVGs can be long — adjust as needed
)

trainer.train()

# Save the LoRA adapter (small file — typically 10-50 MB)
model.save_pretrained("/content/drive/MyDrive/DL Midterm/svg-lora-adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/DL Midterm/svg-lora-adapter")

print("Training loop ready — uncomment and provide your dataset to start training!")

Adding EOS to train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.678586
100,0.503841
150,0.473467
200,0.441511
250,0.428181
300,0.407742
350,0.421935
400,0.396058
450,0.411987
500,0.400586


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training loop ready — uncomment and provide your dataset to start training!


In [ ]:
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)

print("model pad:", model.config.pad_token_id)

pad: <|endoftext|> 151643
bos: None None
eos: <|im_end|> 151645
model pad: 151643


In [ ]:
# ---------- 7. Loading and Merging LoRA Weights ----------
# After training, you can either:
#   (a) Load the adapter on top of the base model
#   (b) Merge the adapter into the base model for faster inference

from peft import PeftModel

# Option (a): Load adapter
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/DL Midterm/svg-lora-adapter")

# Option (b): Merge and unload (no overhead at inference time)
model = model.merge_and_unload()
model.save_pretrained("./svg-model-merged")

print("After training, merge LoRA weights for zero-overhead inference:")
print("  model = model.merge_and_unload()")
print("  # W' = W + (alpha/r) * B @ A  →  single matrix, no adapter overhead")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

After training, merge LoRA weights for zero-overhead inference:
  model = model.merge_and_unload()
  # W' = W + (alpha/r) * B @ A  →  single matrix, no adapter overhead


#Evaluation Code


In [4]:
import os

os.chdir("/content/drive/MyDrive/DL Midterm")

In [5]:
!pip install -q lxml

In [6]:
import io
import re
import torch
import cairosvg
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from PIL import Image
from tqdm import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lxml import etree

# =========================
# CONFIG
# =========================
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = "./svg-lora-adapter"
MERGED_PATH = "./svg-model-merged"

TEST_CSV = "test.csv"            # must contain: id, prompt
SUBMISSION_CSV = "submission.csv"
DEBUG_CSV = "submission_debug.csv"

MAX_NEW_TOKENS = 1536
BATCH_SIZE = 512

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()

print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


# =========================
# SVG HELPERS
# =========================
def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def extract_svg(text: str) -> str:
    text = text.strip()

    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text

def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None

def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()

    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"

        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"

    if render_svg(svg) is None:
        return 0, "render_failed"

    return 1, "valid"

def repair_svg(svg: str) -> str:

    if not svg:
        return svg

    svg = svg.strip()

    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]

    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()

    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[:end + len("</svg>")]

    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"

    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)

    svg = re.sub(r"<[^>]*$", "", svg)

    return svg

def recover_svg_with_lxml(svg: str) -> str:

    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg


def clean_svg_output(raw_text: str, prompt: str):
    svg = extract_svg(raw_text)

    valid, reason = validity_gate(svg)
    if valid == 1:
        return svg, "valid", svg, raw_text

    repaired = repair_svg(svg)
    valid, reason = validity_gate(repaired)
    if valid == 1:
        return repaired, "repaired", svg, raw_text

    xml = recover_svg_with_lxml(repaired)
    valid, reason = validity_gate(xml)
    if valid == 1:
        return xml, "xml", svg, raw_text

    return FALLBACK_SVG, "fallback", svg, raw_text

# =========================
# GENERATION
# =========================
def build_model_input(prompt: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n"

def generate_svg_batch_debug(prompts, batch_size):
    final_svgs = []
    debug_rows = []

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]
        inputs_text = [build_model_input(p) for p in batch_prompts]

        inputs = tokenizer(
            inputs_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)


        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )


        prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
        generated_only = [
            outputs[j, int(prompt_lengths[j]):]
            for j in range(outputs.shape[0])
        ]

        decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

        for prompt, raw in zip(batch_prompts, decoded):
            final_svg, reason, extracted_svg, raw_text = clean_svg_output(raw, prompt)
            final_svgs.append(final_svg)
            debug_rows.append({
                "prompt": prompt,
                "reason": reason,
                "raw_text": raw_text,
                "extracted_svg": extracted_svg,
                "final_svg": final_svg,
            })

    return final_svgs, pd.DataFrame(debug_rows)

# =========================
# FINAL SUBMISSION
# =========================
def build_submission_csv(
    test_csv_path: str = TEST_CSV,
    output_csv_path: str = SUBMISSION_CSV,
    debug_csv_path: str = DEBUG_CSV,
    batch_size: int = BATCH_SIZE,
):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)

    prompts = test_df["prompt"].tolist()
    ids = test_df["id"].tolist()

    svgs, debug_df = generate_svg_batch_debug(prompts, batch_size=BATCH_SIZE)
    print(debug_df["reason"].value_counts())

    assert len(ids) == len(svgs), f"ids={len(ids)}, svgs={len(svgs)}"

    submission_df = pd.DataFrame({
        "id": ids,
        "svg": svgs
    })

    submission_df.to_csv(output_csv_path, index=False)
    debug_df.to_csv("submission_debug.csv", index=False)
    return submission_df


# =========================
# BUILD FINAL SUBMISSION
# =========================
submission_df = build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    batch_size=BATCH_SIZE
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

pad: <|endoftext|> 151643
bos: None None
eos: <|im_end|> 151645
model pad: None


100%|██████████| 2/2 [07:47<00:00, 233.98s/it]


reason
xml         550
valid       449
repaired      1
Name: count, dtype: int64
